# Provision Compute Resources

This notebook creates and configures GPU compute clusters in Azure ML for training small language models.

## What This Notebook Does

1. **Checks Existing Compute**: Lists current compute resources to avoid duplicates
2. **Estimates Costs**: Calculates hourly costs for different VM sizes to help you choose the right configuration
3. **Creates GPU Cluster**: Provisions an auto-scaling compute cluster with GPU nodes
4. **Validates Compute**: Ensures the cluster is ready and accessible for training jobs
5. **Configures Auto-scaling**: Sets minimum and maximum node counts for cost optimization

## Why GPU Compute Matters

- **Training Speed**: GPUs accelerate model fine-tuning by 10-50x compared to CPUs
- **Cost Efficiency**: Auto-scaling reduces costs by scaling down to zero nodes when idle
- **Memory**: Large models require GPU memory for efficient batch processing
- **CUDA Support**: PyTorch and Hugging Face transformers are optimized for NVIDIA GPUs

## Recommended VM Sizes

| VM Size | GPUs | GPU Memory | Use Case | Approx Cost/hr |
|---------|------|------------|----------|----------------|
| Standard_NC6s_v3 | 1x V100 | 16GB | Small models, testing | ~$3.06 |
| Standard_NC12s_v3 | 2x V100 | 32GB | Medium models | ~$6.12 |
| Standard_NC24s_v3 | 4x V100 | 64GB | Large models, faster training | ~$12.24 |

## Auto-scaling Configuration

- **Minimum Nodes**: 0 (scales to zero when idle to save costs)
- **Maximum Nodes**: 4 (limits concurrent training jobs)
- **Idle Time Before Scale Down**: 120 seconds

## Prerequisites

- Completed notebook `01-setup-infrastructure.ipynb`
- Azure ML workspace provisioned
- GPU quota available in your subscription (check Azure Portal → Subscriptions → Usage + quotas)

## Expected Duration

~5-10 minutes for cluster provisioning

## 1. Setup and Configuration

**Why load configuration?** We centralize all Azure and compute settings in YAML config files and `.env` to avoid hardcoding credentials and make the workflow portable across environments (dev/prod).

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.utils.config import load_config
from src.utils.azure_auth import get_ml_client
from src.utils.compute_utils import (
    create_compute_cluster,
    get_compute_status,
    list_compute_clusters,
    wait_for_compute_ready,
    get_available_vm_sizes,
    estimate_compute_cost,
)

print("✅ Imports successful")

In [ ]:
# Load configuration
config = load_config()

print(f"Subscription: {config.azure.subscription_id}")
print(f"Resource Group: {config.azure.resource_group}")
print(f"Workspace: {config.azure.workspace_name}")

In [ ]:
# Connect to Azure ML workspace
ml_client = get_ml_client(config.azure)

print(f"✅ Connected to workspace: {ml_client.workspace_name}")

## 2. Check Existing Compute Resources

**Why check existing resources?** Avoid creating duplicate compute clusters which can lead to unexpected costs and resource management issues. If a cluster already exists, we can reuse it.

In [ ]:
# List all compute resources
computes = list_compute_clusters(ml_client)

print(f"\nFound {len(computes)} compute resource(s):\n")
for compute in computes:
    print(f"  - {compute.name}")
    print(f"    Type: {compute.type}")
    print(f"    State: {compute.provisioning_state}")
    if hasattr(compute, 'size'):
        print(f"    VM Size: {compute.size}")
    print()

## 3. Review Available GPU VM Sizes

In [ ]:
# Get available VM sizes
vm_sizes = get_available_vm_sizes(ml_client)

print("Common GPU VM Sizes:\n")
print(f"{'VM Size':<30} {'GPUs':<6} {'GPU Type':<10} {'RAM (GB)':<10} {'vCPUs':<8}")
print("-" * 70)

for vm in vm_sizes:
    print(f"{vm['name']:<30} {vm['gpus']:<6} {vm['gpu_type']:<10} {vm['ram_gb']:<10} {vm['vcpus']:<8}")

## 4. Configure Compute Cluster

**Important Configuration Choices:**

### VM Size Selection
- **Development**: `Standard_NC6s_v3` (1x V100, 6 vCPUs, 112 GB RAM) - ~$3/hr
- **Production**: `Standard_NC24ads_A100_v4` (1x A100, 24 vCPUs, 220 GB RAM) - ~$4/hr

### Scaling Configuration
- **min_instances**: 0 (scales to zero when idle for cost savings)
- **max_instances**: 4 (adjust based on workload and quota)
- **idle_time**: 300 seconds (5 minutes before scale down)

### Priority Tier
- **dedicated**: Guaranteed capacity, higher cost
- **low_priority**: Lower cost (~80% discount), may be preempted

In [ ]:
# Compute cluster configuration
COMPUTE_NAME = "gpu-cluster"  # Change if needed
VM_SIZE = "Standard_NC6s_v3"  # V100 GPU
MIN_INSTANCES = 0
MAX_INSTANCES = 4
IDLE_TIME = 300  # 5 minutes
TIER = "dedicated"  # or "low_priority"

print("Compute Configuration:")
print(f"  Name: {COMPUTE_NAME}")
print(f"  VM Size: {VM_SIZE}")
print(f"  Min Instances: {MIN_INSTANCES}")
print(f"  Max Instances: {MAX_INSTANCES}")
print(f"  Idle Time: {IDLE_TIME} seconds")
print(f"  Tier: {TIER}")

## 5. Estimate Costs

In [ ]:
# Estimate compute costs
# Assume 8 hours of usage per day (training jobs)
cost_estimate = estimate_compute_cost(
    vm_size=VM_SIZE,
    max_instances=MAX_INSTANCES,
    hours_per_day=8.0
)

print("\nCost Estimate (USD):")
print(f"  VM Size: {cost_estimate['vm_size']}")
print(f"  Hourly Rate (per node): ${cost_estimate['hourly_rate_per_node']:.2f}")
print(f"  Max Instances: {cost_estimate['max_instances']}")
print(f"  Usage: {cost_estimate['hours_per_day']} hours/day")
print(f"\n  Estimated Daily Cost: ${cost_estimate['estimated_daily_cost']:.2f}")
print(f"  Estimated Monthly Cost: ${cost_estimate['estimated_monthly_cost']:.2f}")
print(f"\n  Note: {cost_estimate['note']}")

## 6. Create Compute Cluster

⚠️ **This will provision Azure resources that incur costs!**

The cluster scales to 0 when idle to minimize costs.

In [ ]:
# Create or update compute cluster
compute = create_compute_cluster(
    ml_client=ml_client,
    compute_name=COMPUTE_NAME,
    vm_size=VM_SIZE,
    min_instances=MIN_INSTANCES,
    max_instances=MAX_INSTANCES,
    idle_time_before_scale_down=IDLE_TIME,
    tier=TIER,
)

print(f"\n✅ Compute cluster '{COMPUTE_NAME}' created/updated")
print(f"   ID: {compute.id}")

## 7. Wait for Cluster to be Ready

Provisioning can take 5-10 minutes for new clusters.

In [ ]:
# Wait for compute to be ready (max 10 minutes)
is_ready = wait_for_compute_ready(
    ml_client=ml_client,
    compute_name=COMPUTE_NAME,
    timeout_seconds=600,
    poll_interval=15,
)

if is_ready:
    print(f"\n✅ Compute cluster '{COMPUTE_NAME}' is ready!")
else:
    print(f"\n⚠️  Timeout - check status in Azure ML Studio")

## 8. Verify Cluster Status

In [ ]:
# Get detailed status
status = get_compute_status(ml_client, COMPUTE_NAME)

print("\nCompute Cluster Status:")
print(f"  Name: {status['name']}")
print(f"  Type: {status.get('type', 'N/A')}")
print(f"  State: {status['state']}")
print(f"  VM Size: {status.get('vm_size', 'N/A')}")
print(f"  Min Instances: {status.get('min_instances', 'N/A')}")
print(f"  Max Instances: {status.get('max_instances', 'N/A')}")
print(f"  Current Nodes: {status.get('current_node_count', 0)}")
print(f"  Tier: {status.get('tier', 'N/A')}")

## 9. View in Azure ML Studio

You can also view and manage compute in [Azure ML Studio](https://ml.azure.com):

1. Navigate to your workspace
2. Go to **Compute** → **Compute clusters**
3. Select your cluster to view details, metrics, and logs

In [ ]:
# Generate Azure ML Studio URL
studio_url = (
    f"https://ml.azure.com/compute/list/amlcompute"
    f"?wsid=/subscriptions/{config.azure.subscription_id}"
    f"/resourceGroups/{config.azure.resource_group}"
    f"/providers/Microsoft.MachineLearningServices"
    f"/workspaces/{config.azure.workspace_name}"
)

print("View compute in Azure ML Studio:")
print(studio_url)

## Summary

✅ **Compute cluster successfully provisioned!**

**What's configured:**
- GPU-enabled compute cluster for training
- Auto-scaling from 0 to max instances
- Automatic scale-down after idle period
- System-assigned managed identity

**Cost optimization:**
- Scales to 0 nodes when idle (no compute costs)
- Automatically scales down after 5 minutes of inactivity
- Only pay for compute time during training

## Next Steps

- **Notebook 04**: Download Phi-4 model from Azure AI Foundry
- **Notebook 05**: Submit training job to this compute cluster

## Troubleshooting

**Insufficient Quota Error**:
- Request quota increase: [Azure Portal → Quotas](https://portal.azure.com/#view/Microsoft_Azure_Capacity/QuotaMenuBlade)
- Or try smaller VM size (e.g., Standard_NC6s_v3 instead of NC24)

**Provisioning Failed**:
- Check Azure ML Studio for detailed error messages
- Verify sufficient permissions in subscription
- Ensure VM size is available in your region

**Slow Provisioning**:
- First-time provisioning takes longer (5-10 minutes)
- Subsequent updates are faster
- Cluster remains in "Updating" state during scaling operations